# Notebook 1 — Baselines, intervalos de confianza y efecto de la partición

Evaluación del sistema de traducción automática **qom ↔ español** (tesis de maestría,
FCEN-UBA). Los modelos son fine-tunes de `facebook/nllb-200-distilled-600M` sobre el
corpus paralelo QomL'aqtaq, con la etiqueta `grn_Latn` (guaraní) como **proxy** para qom
y `spa_Latn` para español.

Esta notebook **parte de las predicciones ya generadas** (un CSV con una fila por
segmento traducido) y de un CSV con métricas agregadas a nivel corpus. **No** vuelve a
correr los modelos: todo lo que se calcula acá (chrF++ por segmento, bootstrap, tests)
es barato sobre texto ya existente y no requiere GPU.

### Sistemas del estudio

| Alias | Entrenamiento | Particiones |
|---|---|---|
| `nllb-base` | ninguno (zero-shot) | — |
| `qom-mt-biblia` | solo Biblia (30.449 segmentos) | aleatoria |
| `qom-mt-v1` | QomL-Base: relatos + manual de salud (2.882 segm.) | estratificada y aleatoria |
| `qom-mt-v2` | QomL-Base + Biblia (33.331 segm.) | estratificada y aleatoria |

- **Estratificada**: reparte por documento fuente (cada documento contribuye
  proporcionalmente a train/dev/test).
- **Aleatoria**: asigna los pares uniformemente al azar.

### Test set común

Todas las evaluaciones usan **el test set estratificado de QomL-Base (197 pares)**, en
ambas direcciones (`qom2es` y `es2qom`). Es la única forma de comparar sistemas
entrenados sobre corpus distintos. **No** se usan los test sets propios de cada modelo.

### Estructura

- **Paso 0** — Inventario de lo que ya existe (+ bloque condicional de generación).
- **1.1** — Chequeo de contaminación train/test.
- **1.2** — Métricas a nivel corpus (recalculadas) + chrF++ por segmento.
- **1.3** — Intervalos de confianza (bootstrap pareado + test de permutación).
- **1.4** — Figuras y tablas de salida.

## Celda de configuración

**Completá las rutas marcadas con `RELLENAR`.** Los slots están validados más abajo: si
alguno queda sin completar, la notebook falla con un mensaje claro en vez de adivinar.

- `TRANSLATIONS_CSV` — CSV de traducciones, **una fila por segmento**.
- `CORPUS_METRICS_CSV` — CSV de métricas agregadas a nivel corpus (BLEU/chrF ya calculados).
- `TRAIN_SETS` — por sistema fine-tuneado, el CSV/TSV del set de **entrenamiento** (para el
  chequeo de contaminación de 1.1). `nllb-base` no lleva (es zero-shot).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ── Reproducibilidad ──────────────────────────────────────────────────────────
RANDOM_STATE = 20260729
np.random.seed(RANDOM_STATE)

# ── Rutas de entrada (RELLENAR) ───────────────────────────────────────────────
# Poné acá las rutas a tus CSV. Usá rutas absolutas o relativas a esta notebook.
DATA_DIR = Path("RELLENAR/data")            # carpeta con los CSV de entrada

TRANSLATIONS_CSV   = DATA_DIR / "RELLENAR_traducciones.csv"      # 1 fila por segmento
CORPUS_METRICS_CSV = DATA_DIR / "RELLENAR_metricas_corpus.csv"   # métricas a nivel corpus

# Sets de entrenamiento por sistema fine-tuneado (para contaminación en 1.1).
# Cada CSV/TSV debe tener las columnas de texto qom y español del train de ese sistema.
# 'nllb-base' es zero-shot: no lleva entrada acá.
TRAIN_SETS = {
    "qom-mt-biblia":           DATA_DIR / "RELLENAR_train_biblia.csv",
    "qom-mt-v1-estratificado": DATA_DIR / "RELLENAR_train_v1_estrat.csv",
    "qom-mt-v1-aleatorio":     DATA_DIR / "RELLENAR_train_v1_aleatorio.csv",
    "qom-mt-v2-estratificado": DATA_DIR / "RELLENAR_train_v2_estrat.csv",
    "qom-mt-v2-aleatorio":     DATA_DIR / "RELLENAR_train_v2_aleatorio.csv",
}

# ── Rutas de salida ───────────────────────────────────────────────────────────
RESULTS_DIR = Path("resultados")            # CSV de salida (uno por resultado numérico)
FIG_DIR     = Path("../poster/figures")     # figuras para el póster
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# chrF++ por segmento: lo consume 1.3 y la Notebook 2. Mantené el mismo path en la NB2.
SEGMENT_CHRF_CSV = RESULTS_DIR / "chrf_por_segmento.csv"

# ── Parámetros de generación (idénticos a los ya usados) ──────────────────────
# Sólo se usan si se activa el bloque condicional de generación del Paso 0.
GEN_PARAMS = dict(num_beams=4, no_repeat_ngram_size=3, max_new_tokens=128)
# Etiquetas de idioma NLLB: qom vía proxy grn_Latn (guaraní), como en los fine-tunes.
LANG = {"qom": "grn_Latn", "es": "spa_Latn"}
# direction -> (idioma fuente, idioma referencia)
DIRECTION_LANGS = {"qom2es": ("qom", "es"), "es2qom": ("es", "qom")}

# ── Constantes del estudio ────────────────────────────────────────────────────
N_TEST_ESPERADO = 197        # pares del test estratificado de Base
N_BOOTSTRAP     = 1000       # réplicas del bootstrap (1.3)
N_PERMUTACIONES = 1000       # permutaciones del test pareado (1.3)
SIM_UMBRAL      = 0.80       # umbral de similitud coseno para casi-duplicados (1.1)

# Orden canónico de sistemas (para figuras/tablas). Se filtra a los presentes.
SISTEMAS_ORDEN = [
    "nllb-base",
    "qom-mt-biblia",
    "qom-mt-v1-estratificado", "qom-mt-v1-aleatorio",
    "qom-mt-v2-estratificado", "qom-mt-v2-aleatorio",
]

print("Config cargada.")
print(f"  RANDOM_STATE = {RANDOM_STATE}")
print(f"  Salidas en   : {RESULTS_DIR.resolve()}")
print(f"  Figuras en   : {FIG_DIR.resolve()}")

In [ ]:
# Validación de los slots: fallar claro si algo quedó sin completar.
def _chequear_rellenar():
    faltan = []
    for nombre, ruta in [("TRANSLATIONS_CSV", TRANSLATIONS_CSV),
                         ("CORPUS_METRICS_CSV", CORPUS_METRICS_CSV)]:
        if "RELLENAR" in str(ruta):
            faltan.append(nombre)
    trains_rellenar = [k for k, v in TRAIN_SETS.items() if "RELLENAR" in str(v)]
    if faltan:
        raise ValueError(
            "Completá estas rutas en la celda de configuración antes de seguir: "
            + ", ".join(faltan))
    if trains_rellenar:
        print("[aviso] Estos TRAIN_SETS siguen con 'RELLENAR' y se saltearán en 1.1:")
        for k in trains_rellenar:
            print(f"        - {k}")
    # Existencia de los CSV principales
    for nombre, ruta in [("TRANSLATIONS_CSV", TRANSLATIONS_CSV),
                         ("CORPUS_METRICS_CSV", CORPUS_METRICS_CSV)]:
        if not Path(ruta).exists():
            raise FileNotFoundError(f"No existe {nombre}: {ruta}")

_chequear_rellenar()
print("Rutas principales OK.")

## Paso 0 — Inventario de lo que ya existe

Cargamos los dos CSV, inspeccionamos columnas y tipos, y detectamos los nombres de
columna **sin asumirlos**: cada uno se mapea explícitamente a los nombres canónicos
(`segment_id, direction, system, split, source, reference, hypothesis`). Si algo no se
puede mapear, se falla con un mensaje claro en vez de adivinar.

In [ ]:
import re
import unicodedata

# ── Detección de columnas: candidatos por nombre canónico (todo en minúscula) ──
CANDIDATOS = {
    "segment_id": ["segment_id", "seg_id", "segmentid", "id", "idx", "id_segmento",
                   "sent_id", "sentence_id"],
    "direction":  ["direction", "dir", "sentido", "direccion", "dirección", "par"],
    "system":     ["system", "sistema", "model", "modelo", "alias", "sys"],
    "split":      ["split", "particion", "partición", "partition", "part", "fold"],
    "source":     ["source", "src", "fuente", "entrada", "input", "origen"],
    "reference":  ["reference", "ref", "referencia", "target", "gold"],
    "hypothesis": ["hypothesis", "hyp", "prediction", "pred", "hipotesis", "hipótesis",
                   "traduccion", "traducción", "output", "mt"],
}

def detectar_columnas(df, requeridas, contexto=""):
    lower = {c.lower().strip(): c for c in df.columns}
    mapa = {}
    for canon in requeridas:
        elegido = None
        for cand in CANDIDATOS[canon]:
            if cand in lower:
                elegido = lower[cand]
                break
        if elegido is None:
            raise ValueError(
                f"[{contexto}] No pude mapear la columna canónica '{canon}'. "
                f"Columnas disponibles: {list(df.columns)}. "
                f"Candidatos buscados: {CANDIDATOS[canon]}. "
                f"Renombrá la columna o agregá el nombre a CANDIDATOS['{canon}'].")
        mapa[canon] = elegido
    return mapa

# ── Normalización de direcciones a {'qom2es','es2qom'} ────────────────────────
DIRECTION_ALIASES = {
    "qom2es": "qom2es", "qom-es": "qom2es", "qom_es": "qom2es", "qomes": "qom2es",
    "qom->es": "qom2es", "qom→es": "qom2es", "tob2spa": "qom2es", "tob-spa": "qom2es",
    "qom2spa": "qom2es", "qom-spa": "qom2es",
    "es2qom": "es2qom", "es-qom": "es2qom", "es_qom": "es2qom", "esqom": "es2qom",
    "es->qom": "es2qom", "es→qom": "es2qom", "spa2tob": "es2qom", "spa-tob": "es2qom",
    "spa2qom": "es2qom", "spa-qom": "es2qom",
}

def norm_direction(v):
    k = str(v).strip().lower().replace(" ", "")
    if k in DIRECTION_ALIASES:
        return DIRECTION_ALIASES[k]
    raise ValueError(
        f"Dirección no reconocida: {v!r}. Agregá el alias a DIRECTION_ALIASES "
        f"(claves válidas de salida: 'qom2es', 'es2qom').")

# ── Normalización de texto (para matching de contaminación) ───────────────────
_PUNT = re.compile(r"[^\w\s]", flags=re.UNICODE)
_ESP  = re.compile(r"\s+")

def normalizar_texto(t):
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = _PUNT.sub(" ", t)          # saca puntuación
    t = _ESP.sub(" ", t).strip()   # colapsa espacios
    return t

print("Helpers de detección y normalización listos.")

In [ ]:
# ── Carga del CSV de traducciones + mapeo a nombres canónicos ─────────────────
tr_raw = pd.read_csv(TRANSLATIONS_CSV)
print("CSV de traducciones:")
print(f"  filas x columnas: {tr_raw.shape}")
print(f"  columnas: {list(tr_raw.columns)}")
print(f"  dtypes:\n{tr_raw.dtypes.to_string()}")

mapa_tr = detectar_columnas(
    tr_raw,
    ["segment_id", "direction", "system", "split", "source", "reference", "hypothesis"],
    contexto="traducciones")
print("\nMapeo de columnas (canónico <- original):")
for k, v in mapa_tr.items():
    print(f"  {k:12s} <- {v}")

tr = tr_raw.rename(columns={v: k for k, v in mapa_tr.items()}).copy()
tr = tr[["segment_id", "direction", "system", "split",
         "source", "reference", "hypothesis"]].copy()
tr["direction"] = tr["direction"].map(norm_direction)
# El texto puede venir con NaN si algún sistema no cubre alguna combinación.
for c in ["source", "reference", "hypothesis"]:
    tr[c] = tr[c].astype("string")
print(f"\nTraducciones normalizadas: {tr.shape[0]} filas.")

In [ ]:
# ── Inventario: sistemas, particiones, direcciones, conteos ───────────────────
print("Sistemas presentes :", sorted(tr["system"].unique()))
print("Particiones (split):", sorted(tr["split"].dropna().unique()))
print("Direcciones        :", sorted(tr["direction"].unique()))

conteo = (tr.groupby(["system", "split", "direction"])
            .size().rename("n_segmentos").reset_index()
            .sort_values(["system", "split", "direction"]))
conteo["coincide_197"] = conteo["n_segmentos"] == N_TEST_ESPERADO
print("\nSegmentos por sistema x partición x dirección:")
print(conteo.to_string(index=False))
conteo.to_csv(RESULTS_DIR / "inventario_conteos.csv", index=False)

no_197 = conteo[~conteo["coincide_197"]]
if len(no_197):
    print(f"\n[aviso] {len(no_197)} combinaciones NO tienen {N_TEST_ESPERADO} segmentos. "
          "Revisá si es esperable (p. ej. sistema/dirección incompleto).")

In [ ]:
# ── Carga del CSV de métricas agregadas a nivel corpus ────────────────────────
cm_raw = pd.read_csv(CORPUS_METRICS_CSV)
print("CSV de métricas corpus:")
print(f"  filas x columnas: {cm_raw.shape}")
print(f"  columnas: {list(cm_raw.columns)}")
print(cm_raw.head().to_string(index=False))

# Mapeamos las columnas de identificación que existan (no todas son obligatorias acá).
lower_cm = {c.lower().strip(): c for c in cm_raw.columns}
mapa_cm = {}
for canon in ["system", "split", "direction"]:
    for cand in CANDIDATOS[canon]:
        if cand in lower_cm:
            mapa_cm[canon] = lower_cm[cand]
            break
print("\nMapeo de identificadores en métricas corpus:", mapa_cm)

# Detectar columnas de métrica (bleu / chrf) por nombre.
col_bleu = next((lower_cm[c] for c in lower_cm if "bleu" in c), None)
col_chrf = next((lower_cm[c] for c in lower_cm if "chrf" in c or "chr_f" in c), None)
print(f"Columna BLEU: {col_bleu}")
print(f"Columna chrF: {col_chrf}")

### Sobre el chrF del CSV agregado: ¿chrF++ o chrF a secas?

Sólo **chrF++** (`word_order=2`) es comparable con la literatura de referencia. El chrF por
defecto de sacrebleu usa `word_order=0` (sin n-gramas de palabra). Si el CSV agregado no
documenta con qué `word_order` se calculó, lo tratamos como **dato no verificado**: se usa
únicamente como control de consistencia contra los valores recalculados en 1.2, **no** como
cifra a reportar.

In [ ]:
# ── ¿El chrF del CSV agregado es chrF++ (word_order=2) o chrF (word_order=0)? ──
# Buscamos alguna pista en el nombre de la columna o en columnas de metadatos.
def inferir_word_order(nombre_col, df):
    if nombre_col is None:
        return None, "sin columna chrF en el CSV agregado"
    n = nombre_col.lower()
    if "++" in n or "chrf2" in n or "wo2" in n or "word_order2" in n:
        return 2, f"nombre de columna sugiere chrF++ ('{nombre_col}')"
    if n.strip() in ("chrf", "chr_f") or n.endswith("chrf"):
        # ambiguo: 'chrf' podría ser default (0) o ++ mal nombrado
        return None, (f"nombre '{nombre_col}' ambiguo: no puedo determinar word_order")
    # ¿hay una columna explícita 'word_order'?
    for c in df.columns:
        if c.lower().strip() in ("word_order", "chrf_word_order", "wo"):
            vals = sorted(pd.unique(df[c].dropna()))
            if len(vals) == 1:
                return int(vals[0]), f"columna '{c}' = {vals[0]}"
    return None, f"no determinable a partir de '{nombre_col}'"

CHRF_WORD_ORDER_AGREGADO, motivo_wo = inferir_word_order(col_chrf, cm_raw)
print(f"word_order del chrF agregado: {CHRF_WORD_ORDER_AGREGADO}  ({motivo_wo})")
if CHRF_WORD_ORDER_AGREGADO != 2:
    print("[aviso] El chrF agregado NO está confirmado como chrF++ (word_order=2).")
    print("        Se usará sólo como control de consistencia, no como cifra a reportar.")

In [ ]:
# ── Consistencia de segment_id entre ambos CSV ────────────────────────────────
# El CSV agregado es a nivel corpus, así que puede no tener segment_id. Chequeamos
# consistencia sólo si la columna existe.
if "segment_id" in {c.lower().strip() for c in cm_raw.columns}:
    lower_cm2 = {c.lower().strip(): c for c in cm_raw.columns}
    ids_tr = set(tr["segment_id"].astype(str))
    ids_cm = set(cm_raw[lower_cm2["segment_id"]].astype(str))
    solo_tr = ids_tr - ids_cm
    solo_cm = ids_cm - ids_tr
    print(f"segment_id en traducciones: {len(ids_tr)} | en métricas: {len(ids_cm)}")
    print(f"  sólo en traducciones: {len(solo_tr)} | sólo en métricas: {len(solo_cm)}")
    if solo_tr or solo_cm:
        print("[aviso] Los segment_id no coinciden del todo entre ambos CSV.")
else:
    print("El CSV de métricas es a nivel corpus (sin segment_id): no aplica el cruce por id.")

### Bloque condicional — generación de traducciones faltantes

Si del inventario surge que falta alguna combinación (típicamente `nllb-base` zero-shot o
las variantes estratificadas), esta sección las genera y las agrega a `tr`, respetando:

- parámetros de generación **idénticos** a los ya usados (`GEN_PARAMS`, declarados en la
  config);
- para el zero-shot, las mismas etiquetas de idioma que los fine-tunes
  (`grn_Latn` / `spa_Latn`).

**Este es el único bloque que puede requerir `transformers` y (idealmente) GPU.** Si no
falta nada, se saltea solo. Completá `MODELOS_A_GENERAR` con los sistemas faltantes y sus
checkpoints; el resto de la notebook no lo necesita.

In [ ]:
# ── ¿Qué combinaciones esperamos y cuáles faltan? ─────────────────────────────
# Definí acá qué sistemas esperás tener sobre el test estratificado de Base.
SISTEMAS_ESPERADOS = set(SISTEMAS_ORDEN)
DIRECCIONES = ["qom2es", "es2qom"]

presentes = set(zip(tr["system"], tr["direction"]))
faltantes = [(s, d) for s in SISTEMAS_ESPERADOS for d in DIRECCIONES
             if (s, d) not in presentes]

print("Combinaciones (sistema, dirección) faltantes respecto de lo esperado:")
if faltantes:
    for s, d in sorted(faltantes):
        print(f"  - {s} / {d}")
else:
    print("  (ninguna) — no hace falta generar nada, el bloque siguiente se saltea.")

In [ ]:
# ── Generación de las combinaciones faltantes (sólo si hace falta) ────────────
# Completá el mapeo sistema -> {'qom2es': checkpoint, 'es2qom': checkpoint}.
# Para nllb-base (zero-shot) usá 'facebook/nllb-200-distilled-600M' en ambas direcciones.
MODELOS_A_GENERAR = {
    # "nllb-base": {"qom2es": "facebook/nllb-200-distilled-600M",
    #               "es2qom": "facebook/nllb-200-distilled-600M"},
    # "qom-mt-v2-estratificado": {"qom2es": "RELLENAR/checkpoint-qom2es",
    #                             "es2qom": "RELLENAR/checkpoint-es2qom"},
}

def _test_pairs_base():
    # Reconstruye los pares (segment_id, qom, es) del test estratificado de Base a partir
    # de cualquier sistema ya presente que cubra ese test. Sirve como fuente para generar.
    base = tr[tr["split"].astype(str).str.contains("estrat", case=False, na=False)]
    if base.empty:
        base = tr
    # Preferimos una dirección para leer ambos lados:
    q2e = base[base["direction"] == "qom2es"][["segment_id", "source", "reference"]]
    q2e = q2e.rename(columns={"source": "qom", "reference": "es"})
    return q2e.drop_duplicates("segment_id").reset_index(drop=True)

if faltantes and MODELOS_A_GENERAR:
    import torch
    from transformers import AutoModelForSeq2SeqLM, NllbTokenizer

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Generando faltantes en {DEVICE}. GEN_PARAMS = {GEN_PARAMS}")

    pares = _test_pairs_base()
    print(f"Test de referencia para generar: {len(pares)} pares.")

    def _load(model_name):
        tok = NllbTokenizer.from_pretrained(model_name)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE).eval()
        return mdl, tok

    def _translate(mdl, tok, textos, src_lang, tgt_lang, batch=8):
        tok.src_lang = src_lang
        forced = tok.convert_tokens_to_ids(tgt_lang)
        out_all = []
        for i in range(0, len(textos), batch):
            b = [str(x) for x in textos[i:i + batch]]
            enc = tok(b, return_tensors="pt", padding=True, truncation=True,
                      max_length=GEN_PARAMS["max_new_tokens"]).to(DEVICE)
            with torch.no_grad():
                gen = mdl.generate(**enc, forced_bos_token_id=forced, **GEN_PARAMS)
            out_all += tok.batch_decode(gen, skip_special_tokens=True)
        return out_all

    nuevas = []
    for (sistema, direccion) in faltantes:
        if sistema not in MODELOS_A_GENERAR:
            print(f"  [skip] {sistema}/{direccion}: sin checkpoint en MODELOS_A_GENERAR.")
            continue
        ckpt = MODELOS_A_GENERAR[sistema][direccion]
        src_l, ref_l = DIRECTION_LANGS[direccion]
        mdl, tok = _load(ckpt)
        srcs = pares[src_l].tolist()
        refs = pares[ref_l].tolist()
        hyps = _translate(mdl, tok, srcs, LANG[src_l], LANG[ref_l])
        nuevas.append(pd.DataFrame({
            "segment_id": pares["segment_id"], "direction": direccion,
            "system": sistema, "split": "estratificado",
            "source": srcs, "reference": refs, "hypothesis": hyps}))
        del mdl, tok
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    if nuevas:
        tr = pd.concat([tr] + nuevas, ignore_index=True)
        # Persistimos el CSV de traducciones ampliado para no regenerar la próxima vez.
        tr.to_csv(RESULTS_DIR / "traducciones_ampliado.csv", index=False)
        print(f"Agregadas {sum(len(x) for x in nuevas)} filas. tr ahora: {tr.shape}")
else:
    print("Nada que generar (o MODELOS_A_GENERAR vacío). Se saltea la generación.")

## 1.1 — Chequeo de contaminación

**Correr antes de interpretar cualquier resultado.** Comparamos los 197 pares del test
estratificado de Base contra el train de cada sistema fine-tuneado, en tres niveles:

1. **Match exacto de par** (texto normalizado: minúsculas, sin puntuación, espacios
   colapsados, NFC).
2. **Match exacto de un solo lado** (el qom o el español por separado).
3. **Casi-duplicados**: TF-IDF de n-gramas de caracteres (3–5) + similitud coseno; se
   reporta todo par con similitud ≥ 0,8 y se exporta a CSV para revisión manual.

Esperamos contaminación **alta** en las variantes de partición aleatoria (esperable por
construcción) y **nula o mínima** en `qom-mt-v1-estratificado` y `qom-mt-biblia`. El caso a
mirar con atención es `qom-mt-v2-estratificado`.

In [ ]:
# ── Test estratificado de Base: un lado qom y un lado es por segment_id ────────
def construir_test_base(tr):
    # Tomamos las filas del test estratificado. Si el split no distingue 'estrat',
    # usamos todas las filas disponibles del test común.
    mask = tr["split"].astype(str).str.contains("estrat", case=False, na=False)
    base = tr[mask] if mask.any() else tr
    # De qom2es sacamos source=qom, reference=es.
    q2e = (base[base["direction"] == "qom2es"][["segment_id", "source", "reference"]]
           .rename(columns={"source": "qom", "reference": "es"}))
    # Si faltara qom2es, lo reconstruimos desde es2qom.
    e2q = (base[base["direction"] == "es2qom"][["segment_id", "source", "reference"]]
           .rename(columns={"source": "es", "reference": "qom"}))
    test = pd.concat([q2e, e2q], ignore_index=True).drop_duplicates("segment_id")
    test = test.dropna(subset=["qom", "es"]).reset_index(drop=True)
    test["qom_norm"] = test["qom"].map(normalizar_texto)
    test["es_norm"]  = test["es"].map(normalizar_texto)
    return test

test_base = construir_test_base(tr)
print(f"Test de Base reconstruido: {test_base.shape[0]} pares "
      f"(esperado ~{N_TEST_ESPERADO}).")
test_base.head(3)

In [ ]:
# ── Carga de un train set y detección de sus columnas qom/es ──────────────────
def cargar_train(ruta):
    sep = "\t" if str(ruta).endswith((".tsv", ".txt")) else ","
    df = pd.read_csv(ruta, sep=sep)
    lower = {c.lower().strip(): c for c in df.columns}
    col_qom = next((lower[c] for c in lower
                    if c in ("qom", "tob", "toba", "qom_text", "src_qom")), None)
    col_es = next((lower[c] for c in lower
                   if c in ("es", "spa", "esp", "espanol", "español", "es_text",
                            "castellano")), None)
    if col_qom is None or col_es is None:
        raise ValueError(
            f"No pude detectar columnas qom/es en {ruta}. Columnas: {list(df.columns)}")
    out = df[[col_qom, col_es]].rename(columns={col_qom: "qom", col_es: "es"}).copy()
    out["qom_norm"] = out["qom"].map(normalizar_texto)
    out["es_norm"]  = out["es"].map(normalizar_texto)
    return out

In [ ]:
# ── Cálculo de contaminación por sistema (3 niveles) ──────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def contaminacion_sistema(test_base, train):
    n = len(test_base)
    train_pairs = set(zip(train["qom_norm"], train["es_norm"]))
    train_qom = set(train["qom_norm"])
    train_es  = set(train["es_norm"])

    # Nivel 1: par exacto.
    m_par = test_base.apply(
        lambda r: (r["qom_norm"], r["es_norm"]) in train_pairs, axis=1)
    # Nivel 2: un solo lado exacto.
    m_qom = test_base["qom_norm"].isin(train_qom)
    m_es  = test_base["es_norm"].isin(train_es)
    m_lado = (m_qom | m_es)

    # Nivel 3: casi-duplicados por TF-IDF char n-gramas (3-5) + coseno.
    def near_dup(lado):
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
        corpus = list(train[f"{lado}_norm"]) + list(test_base[f"{lado}_norm"])
        X = vec.fit_transform(corpus)
        Xtr, Xte = X[:len(train)], X[len(train):]
        sims = linear_kernel(Xte, Xtr)          # tf-idf L2-normalizado -> coseno
        best_idx = sims.argmax(axis=1)
        best_sim = sims.max(axis=1)
        return best_sim, best_idx

    sim_qom, idx_qom = near_dup("qom")
    sim_es,  idx_es  = near_dup("es")
    sim_max = np.maximum(sim_qom, sim_es)
    m_near = sim_max >= SIM_UMBRAL

    detalle = test_base[["segment_id", "qom", "es"]].copy()
    detalle["par_exacto"]   = m_par.values
    detalle["lado_exacto"]  = m_lado.values
    detalle["sim_qom"] = np.round(sim_qom, 3)
    detalle["sim_es"]  = np.round(sim_es, 3)
    detalle["sim_max"] = np.round(sim_max, 3)
    detalle["casi_dup"] = m_near
    # Mejor match de train para revisión manual (sobre el lado que dispara la sim).
    usa_qom = sim_qom >= sim_es
    detalle["train_match"] = np.where(
        usa_qom, train["qom"].values[idx_qom], train["es"].values[idx_es])

    resumen = {
        "n_test": n,
        "par_exacto": int(m_par.sum()),
        "lado_exacto": int(m_lado.sum()),
        "casi_dup(>= %.2f)" % SIM_UMBRAL: int(m_near.sum()),
        "contaminado_algun_nivel": int((m_par | m_lado | m_near).sum()),
    }
    contaminado_ids = set(detalle.loc[m_par | m_lado.values | m_near, "segment_id"])
    return resumen, detalle, contaminado_ids

In [ ]:
# ── Correr contaminación para cada sistema con train disponible ───────────────
filas_resumen = []
contaminados_por_sistema = {}   # system -> set(segment_id)
detalles_near = []

for sistema, ruta in TRAIN_SETS.items():
    if "RELLENAR" in str(ruta):
        print(f"[skip] {sistema}: TRAIN_SET sin completar.")
        continue
    if not Path(ruta).exists():
        print(f"[aviso] {sistema}: no existe {ruta} -> contaminación no calculada (NaN).")
        filas_resumen.append({"system": sistema, "n_test": len(test_base),
                              "par_exacto": np.nan, "lado_exacto": np.nan,
                              f"casi_dup(>= {SIM_UMBRAL:.2f})": np.nan,
                              "contaminado_algun_nivel": np.nan})
        continue
    train = cargar_train(ruta)
    resumen, detalle, cont_ids = contaminacion_sistema(test_base, train)
    resumen = {"system": sistema, **resumen}
    filas_resumen.append(resumen)
    contaminados_por_sistema[sistema] = cont_ids
    d = detalle[detalle["casi_dup"] | detalle["par_exacto"] | detalle["lado_exacto"]].copy()
    d.insert(0, "system", sistema)
    detalles_near.append(d)
    print(f"[ok] {sistema}: {resumen}")

tabla_contaminacion = pd.DataFrame(filas_resumen)
print("\n== Tabla de contaminación por sistema ==")
print(tabla_contaminacion.to_string(index=False))
tabla_contaminacion.to_csv(RESULTS_DIR / "contaminacion_resumen.csv", index=False)

if detalles_near:
    revisar = pd.concat(detalles_near, ignore_index=True).sort_values(
        ["system", "sim_max"], ascending=[True, False])
    revisar.to_csv(RESULTS_DIR / "contaminacion_casos_para_revisar.csv", index=False)
    print(f"\nCasos para revisión manual -> "
          f"{RESULTS_DIR / 'contaminacion_casos_para_revisar.csv'} ({len(revisar)} filas)")

In [ ]:
# ── Test "descontaminado": ítems sin ningún match, por sistema ────────────────
# Para reportar métricas sobre el test completo y sobre el descontaminado.
def ids_descontaminados(sistema):
    cont = contaminados_por_sistema.get(sistema, set())
    return set(test_base["segment_id"]) - cont

resumen_descontam = []
for sistema in contaminados_por_sistema:
    limpios = ids_descontaminados(sistema)
    resumen_descontam.append({"system": sistema,
                              "n_total": len(test_base),
                              "n_descontaminado": len(limpios)})
if resumen_descontam:
    print(pd.DataFrame(resumen_descontam).to_string(index=False))
else:
    print("Sin sistemas con train disponible: no hay descontaminación por sistema.")

## 1.2 — Métricas a nivel corpus

**chrF++ a nivel corpus NO es el promedio de los chrF++ por segmento**: sacrebleu agrega
estadísticas de n-gramas, no puntajes. Recalculamos desde el CSV de traducciones con
`sacrebleu`, para cada sistema × partición × dirección:

- **chrF++** (`CHRF(word_order=2)`) — métrica principal;
- **BLEU** — secundaria (se reporta pero no sostiene conclusiones).

Además calculamos **chrF++ a nivel segmento** para todos los pares y lo guardamos en
`chrf_por_segmento.csv` (lo consumen 1.3 y la Notebook 2).

> Se espera **BLEU cercano a cero** en el zero-shot, y que en ese rango **no discrimine**.

In [ ]:
from sacrebleu.metrics import CHRF, BLEU

chrf_pp = CHRF(word_order=2)   # chrF++
bleu    = BLEU()

def _validos(sub):
    m = sub["hypothesis"].notna() & sub["reference"].notna()
    return sub[m]

# ── Métricas a nivel corpus ───────────────────────────────────────────────────
filas_corpus = []
for (sistema, split, direccion), sub in tr.groupby(["system", "split", "direction"]):
    sub = _validos(sub)
    if sub.empty:
        continue
    hyps = sub["hypothesis"].astype(str).tolist()
    refs = sub["reference"].astype(str).tolist()
    filas_corpus.append({
        "system": sistema, "split": split, "direction": direccion,
        "n": len(sub),
        "chrf_pp": chrf_pp.corpus_score(hyps, [refs]).score,
        "bleu":    bleu.corpus_score(hyps, [refs]).score,
    })

metricas_corpus = pd.DataFrame(filas_corpus).sort_values(
    ["system", "split", "direction"]).reset_index(drop=True)
metricas_corpus[["chrf_pp", "bleu"]] = metricas_corpus[["chrf_pp", "bleu"]].round(2)
print(metricas_corpus.to_string(index=False))
metricas_corpus.to_csv(RESULTS_DIR / "metricas_corpus_recalculadas.csv", index=False)

In [ ]:
# ── chrF++ a nivel segmento (para bootstrap y Notebook 2) ─────────────────────
filas_seg = []
for _, r in tr.iterrows():
    if pd.isna(r["hypothesis"]) or pd.isna(r["reference"]):
        continue
    s = chrf_pp.sentence_score(str(r["hypothesis"]), [str(r["reference"])]).score
    filas_seg.append({
        "segment_id": r["segment_id"], "direction": r["direction"],
        "system": r["system"], "split": r["split"],
        "chrf_segment": round(s, 4),
    })
chrf_segmento = pd.DataFrame(filas_seg)
chrf_segmento.to_csv(SEGMENT_CHRF_CSV, index=False)
print(f"chrF++ por segmento -> {SEGMENT_CHRF_CSV}  ({len(chrf_segmento)} filas)")
print(chrf_segmento.head().to_string(index=False))

In [ ]:
# ── Comparación contra el CSV de métricas agregadas ya existente ──────────────
# Sólo si podemos alinear por (system, split, direction). Si el chrF agregado no está
# confirmado como chrF++, esto es control de consistencia, no validación.
if {"system", "split", "direction"}.issubset(mapa_cm.keys()) and col_chrf is not None:
    cm = cm_raw.rename(columns={
        mapa_cm["system"]: "system", mapa_cm["split"]: "split",
        mapa_cm["direction"]: "direction", col_chrf: "chrf_agregado"})
    if col_bleu:
        cm = cm.rename(columns={col_bleu: "bleu_agregado"})
    cm["direction"] = cm["direction"].map(norm_direction)
    comp = metricas_corpus.merge(
        cm[["system", "split", "direction", "chrf_agregado"]
           + (["bleu_agregado"] if col_bleu else [])],
        on=["system", "split", "direction"], how="left")
    comp["dif_chrf"] = (comp["chrf_pp"] - comp["chrf_agregado"]).round(2)
    print(comp.to_string(index=False))
    comp.to_csv(RESULTS_DIR / "comparacion_corpus_vs_agregado.csv", index=False)
    grandes = comp[comp["dif_chrf"].abs() > 1.0]
    if len(grandes):
        print("\n[aviso] Diferencias > 1 punto chrF. Causa probable: distinto word_order "
              "(chrF vs chrF++), tokenización o normalización previa del texto.")
    if CHRF_WORD_ORDER_AGREGADO != 2:
        print("[nota] El chrF agregado no está confirmado como chrF++: la diferencia "
              "puede deberse sólo a eso. Se reportan los valores recalculados de arriba.")
else:
    print("No se puede alinear el CSV agregado por (system, split, direction): "
          "se omite la comparación. Se reportan los valores recalculados.")

## 1.3 — Intervalos de confianza

**Es la parte más importante de la notebook.** Con n=197, las diferencias reportadas en la
literatura pueden no ser significativas.

- **Bootstrap de remuestreo pareado** sobre los segmentos, `N_BOOTSTRAP` réplicas, IC 95%
  para chrF++ de cada sistema. **Se recalcula la métrica a nivel corpus en cada réplica**
  (no se promedian puntajes de segmento). El remuestreo es *pareado*: las mismas posiciones
  se usan para todos los sistemas, para que las diferencias sean comparables.
- Para cada par de sistemas: IC de la **diferencia** y **p-valor por test de permutación
  pareado**.
- `random_state` fijo (`RANDOM_STATE`).

> El costo real es recalcular chrF++ de corpus `N_BOOTSTRAP` × (nº de grupos) veces. Con
> n≈197 es perfectamente manejable. Si quisieras abaratarlo con la versión promediada de
> puntajes de segmento, habría que **decirlo explícitamente** — acá **no** se hace: se usa
> la agregación correcta de sacrebleu.

In [ ]:
# ── Índices de bootstrap compartidos (remuestreo pareado) ─────────────────────
# Trabajamos por dirección: alineamos los sistemas por segment_id sobre el test común.
rng = np.random.default_rng(RANDOM_STATE)

def matriz_hyp_ref(direccion, split_filter="estrat"):
    # Devuelve, por dirección: dict system -> (hyps, refs) alineados por segment_id,
    # y la lista de segment_id común a todos los sistemas.
    sub = tr[tr["direction"] == direccion]
    m = sub["split"].astype(str).str.contains(split_filter, case=False, na=False)
    sub = sub[m] if m.any() else sub
    sub = _validos(sub)
    # segment_id presentes en todos los sistemas
    por_sys = {s: g.set_index("segment_id") for s, g in sub.groupby("system")}
    if not por_sys:
        return {}, []
    ids_comunes = set.intersection(*[set(g.index) for g in por_sys.values()])
    ids_comunes = sorted(ids_comunes)
    datos = {}
    for s, g in por_sys.items():
        g = g.loc[ids_comunes]
        datos[s] = (g["hypothesis"].astype(str).tolist(),
                    g["reference"].astype(str).tolist())
    return datos, ids_comunes

def bootstrap_indices(n, n_rep):
    return [rng.integers(0, n, size=n) for _ in range(n_rep)]

In [ ]:
# ── IC 95% de chrF++ por sistema (bootstrap pareado, recomputando corpus) ─────
def chrf_corpus(hyps, refs, idx=None):
    if idx is not None:
        hyps = [hyps[i] for i in idx]
        refs = [refs[i] for i in idx]
    return chrf_pp.corpus_score(hyps, [refs]).score

filas_ic = []
boot_scores = {}   # (direction, system) -> np.array de réplicas (para diferencias)
for direccion in ["qom2es", "es2qom"]:
    datos, ids = matriz_hyp_ref(direccion)
    if not datos:
        continue
    n = len(ids)
    idxs = bootstrap_indices(n, N_BOOTSTRAP)   # compartidos entre sistemas (pareado)
    for sistema, (hyps, refs) in datos.items():
        punt = chrf_corpus(hyps, refs)
        reps = np.array([chrf_corpus(hyps, refs, ix) for ix in idxs])
        lo, hi = np.percentile(reps, [2.5, 97.5])
        boot_scores[(direccion, sistema)] = reps
        filas_ic.append({"direction": direccion, "system": sistema, "n": n,
                         "chrf_pp": round(punt, 2),
                         "ic_low": round(lo, 2), "ic_high": round(hi, 2)})

ic_sistemas = pd.DataFrame(filas_ic).sort_values(["direction", "system"])
print(ic_sistemas.to_string(index=False))
ic_sistemas.to_csv(RESULTS_DIR / "ic_chrf_por_sistema.csv", index=False)

In [ ]:
# ── IC de la diferencia + p-valor por test de permutación pareado ─────────────
from itertools import combinations

def permutation_test_pareado(hyps_a, refs_a, hyps_b, refs_b, n_perm, rng):
    # Observado: diferencia de chrF++ de corpus entre A y B (mismos segmentos).
    obs = chrf_corpus(hyps_a, refs_a) - chrf_corpus(hyps_b, refs_b)
    n = len(hyps_a)
    mayores = 0
    for _ in range(n_perm):
        swap = rng.random(n) < 0.5      # por segmento, intercambia A<->B
        ha = [hyps_b[i] if swap[i] else hyps_a[i] for i in range(n)]
        ra = [refs_b[i] if swap[i] else refs_a[i] for i in range(n)]
        hb = [hyps_a[i] if swap[i] else hyps_b[i] for i in range(n)]
        rb = [refs_a[i] if swap[i] else refs_b[i] for i in range(n)]
        dif = chrf_corpus(ha, ra) - chrf_corpus(hb, rb)
        if abs(dif) >= abs(obs):
            mayores += 1
    p = (mayores + 1) / (n_perm + 1)    # estimador conservador
    return obs, p

rng_perm = np.random.default_rng(RANDOM_STATE + 1)
filas_dif = []
for direccion in ["qom2es", "es2qom"]:
    datos, ids = matriz_hyp_ref(direccion)
    if not datos:
        continue
    sistemas = [s for s in SISTEMAS_ORDEN if s in datos]
    for a, b in combinations(sistemas, 2):
        # IC de la diferencia desde el bootstrap pareado ya calculado.
        ra_, rb_ = boot_scores[(direccion, a)], boot_scores[(direccion, b)]
        dif_boot = ra_ - rb_
        lo, hi = np.percentile(dif_boot, [2.5, 97.5])
        obs, p = permutation_test_pareado(*datos[a], *datos[b], N_PERMUTACIONES, rng_perm)
        filas_dif.append({
            "direction": direccion, "sistema_a": a, "sistema_b": b,
            "dif_chrf(a-b)": round(obs, 2),
            "dif_ic_low": round(lo, 2), "dif_ic_high": round(hi, 2),
            "p_permutacion": round(p, 4),
            "significativo_0.05": p < 0.05,
        })

dif_sistemas = pd.DataFrame(filas_dif)
if len(dif_sistemas):
    print(dif_sistemas.to_string(index=False))
    dif_sistemas.to_csv(RESULTS_DIR / "ic_diferencias_y_permutacion.csv", index=False)
else:
    print("No hay pares de sistemas comparables por dirección.")

## 1.4 — Figuras y tablas de salida

- **Figura A** — barras horizontales de chrF++ con barras de error (IC 95%), agrupadas por
  dirección. Línea vertical punteada en el nivel del zero-shot ("piso de la métrica"). BLEU
  como número pequeño al final de cada barra.
- **Tabla B** — estratificado vs aleatorio para `qom-mt-v1` y `qom-mt-v2`, con la columna de
  ítems contaminados de 1.1. Objetivo: mostrar que la ventaja aparente de las variantes
  aleatorias se explica por solapamiento train/test, no por mejor calidad.

Figuras en **PDF vectorial** y **PNG a 300 dpi**. Cada número que aparece en una figura
queda también en un CSV.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Estilo para póster: fuentes grandes, legible a dos metros.
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 18, "axes.titlesize": 22, "axes.labelsize": 20,
    "xtick.labelsize": 16, "ytick.labelsize": 16, "legend.fontsize": 16,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.constrained_layout.use": True,
})

# Paleta Okabe-Ito (apta para daltonismo).
OKABE_ITO = {
    "negro": "#000000", "naranja": "#E69F00", "celeste": "#56B4E9",
    "verde": "#009E73", "amarillo": "#F0E442", "azul": "#0072B2",
    "bermellon": "#D55E00", "violeta": "#CC79A7",
}
COLOR_SISTEMA = {
    "nllb-base": OKABE_ITO["negro"],
    "qom-mt-biblia": OKABE_ITO["naranja"],
    "qom-mt-v1-estratificado": OKABE_ITO["azul"],
    "qom-mt-v1-aleatorio": OKABE_ITO["celeste"],
    "qom-mt-v2-estratificado": OKABE_ITO["verde"],
    "qom-mt-v2-aleatorio": OKABE_ITO["bermellon"],
}

def guardar(fig, nombre):
    fig.savefig(FIG_DIR / f"{nombre}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{nombre}.png", bbox_inches="tight", dpi=300)
    print(f"  guardada: {FIG_DIR / (nombre + '.pdf')}  y  .png (300 dpi)")

In [ ]:
# ── Figura A: chrF++ con IC 95% por sistema, un panel por dirección ───────────
# Para la figura principal preferimos las variantes ESTRATIFICADAS donde existan.
def elegir_variante_estrat(df):
    # Colapsa v1/v2 a su variante estratificada si está; deja el resto igual.
    return df[~df["system"].str.contains("aleatorio", na=False)]

bleu_lookup = metricas_corpus.set_index(
    ["system", "split", "direction"])["bleu"].to_dict()

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True)
for ax, direccion in zip(axes, ["qom2es", "es2qom"]):
    d = ic_sistemas[ic_sistemas["direction"] == direccion].copy()
    d = elegir_variante_estrat(d)
    orden = [s for s in SISTEMAS_ORDEN if s in set(d["system"])]
    d = d.set_index("system").loc[orden].reset_index()

    y = np.arange(len(d))
    err = np.vstack([d["chrf_pp"] - d["ic_low"], d["ic_high"] - d["chrf_pp"]])
    colores = [COLOR_SISTEMA.get(s, OKABE_ITO["violeta"]) for s in d["system"]]
    ax.barh(y, d["chrf_pp"], xerr=err, color=colores, capsize=5,
            error_kw=dict(ecolor="#333333", lw=1.5))
    ax.set_yticks(y)
    ax.set_yticklabels(d["system"])
    ax.invert_yaxis()
    ax.set_title(direccion)
    ax.set_xlabel("chrF++")

    # Piso de la métrica: nivel del zero-shot (nllb-base).
    base = d[d["system"] == "nllb-base"]
    if len(base):
        ax.axvline(base["chrf_pp"].iloc[0], ls=":", color=OKABE_ITO["negro"], lw=2)
        ax.text(base["chrf_pp"].iloc[0], len(d) - 0.4, " piso de la métrica",
                rotation=90, va="bottom", ha="left", fontsize=13, color="#333333")

    # BLEU como número chico al final de cada barra.
    for yi, (_, row) in zip(y, d.iterrows()):
        b = bleu_lookup.get((row["system"], "estratificado", direccion))
        if b is None:  # buscar cualquier split de ese sistema/dirección
            cand = [v for (s, sp, di), v in bleu_lookup.items()
                    if s == row["system"] and di == direccion]
            b = cand[0] if cand else None
        if b is not None:
            ax.text(row["chrf_pp"] + err[1][yi] + 0.5, yi, f"BLEU {b:.1f}",
                    va="center", fontsize=12, color="#555555")

fig.suptitle("chrF++ por sistema con IC 95% (test estratificado de Base, n≈197)")
guardar(fig, "figuraA_chrf_ic")
plt.show()

In [ ]:
# ── Tabla B: estratificado vs aleatorio, con contaminación ────────────────────
cont_col = f"casi_dup(>= {SIM_UMBRAL:.2f})"
cont_lookup = tabla_contaminacion.set_index("system") if len(tabla_contaminacion) else None

filas_b = []
for base_sys in ["qom-mt-v1", "qom-mt-v2"]:
    for variante in ["estratificado", "aleatorio"]:
        sistema = f"{base_sys}-{variante}"
        for direccion in ["qom2es", "es2qom"]:
            fila = ic_sistemas[(ic_sistemas["system"] == sistema) &
                               (ic_sistemas["direction"] == direccion)]
            if fila.empty:
                continue
            r = fila.iloc[0]
            n_cont = np.nan
            if cont_lookup is not None and sistema in cont_lookup.index:
                n_cont = cont_lookup.loc[sistema, "contaminado_algun_nivel"]
            filas_b.append({
                "base": base_sys, "variante": variante, "direction": direccion,
                "chrf_pp": r["chrf_pp"], "ic_low": r["ic_low"], "ic_high": r["ic_high"],
                "items_contaminados": n_cont,
            })

tabla_b = pd.DataFrame(filas_b).sort_values(["base", "direction", "variante"])
print(tabla_b.to_string(index=False))
tabla_b.to_csv(RESULTS_DIR / "tablaB_estrat_vs_aleatorio.csv", index=False)

### Cierre

- La **métrica principal** es chrF++; BLEU se reporta pero no sostiene conclusiones (y
  cerca de cero, en el zero-shot, no discrimina).
- Antes de leer cualquier ventaja, mirar la **Tabla B**: si una variante aleatoria "gana"
  pero tiene ítems contaminados, esa ventaja es solapamiento train/test, no calidad.
- Todos los números que aparecen en las figuras están también en los CSV de `resultados/`.
- El CSV `chrf_por_segmento.csv` es la entrada de la **Notebook 2**.